# Phase 07 — ATS Friendliness Scoring

**Status:** Complete  
**Workflow:** Notebook-first training documentation  
**Purpose:** Design ATS friendliness scoring so CV quality is evaluated separately from job-fit alignment.

This notebook is the Phase 7 source of truth. It writes `reports/phase_07_ats_friendliness_scoring.json` with benchmark coverage, issue taxonomy, transparent rule baseline, evaluation policy, output contract, and current implementation decision.

## Purpose
Document and verify Phase 07 — ATS Friendliness Scoring in the Bisakerja notebook-first training workflow.

## Required input
Use the repository-root training data, artifacts, and reports referenced by this phase.

## Action
Run or review the Phase 07.ats.friendliness.scoring notebook cells in numeric order, preserving generated evidence under reports/ and artifacts/.

## Expected output
Produce or preserve the phase-specific report and artifact evidence for Phase 07 — ATS Friendliness Scoring.

## Verification
Confirm the notebook has no saved error outputs, no unintended unexecuted production code cells, and matching durable report evidence.

## Contract boundary

ATS friendliness measures CV parseability, structure, contact/date evidence, quantified impact evidence, and formatting risk. It does not measure whether a candidate matches a job.

The model/core output owns `atsFriendliness.score` and `atsFriendliness.detectedIssues`. The public API currently exposes `atsFriendliness.score` and `atsFriendliness.summary`; the API wrapper may render `summary` from core ATS issues, but it must not invent unsupported claims or mix job-fit evidence into ATS scoring.

## Shared setup

### Purpose
Load prior label policy and the generated OpenAPI contract, then define helpers used by every Phase 7 step.

### Required input
Repository root with `TODOS.md`, `reports/phase_02_label_schema_baselines.json`, `reports/phase_03_normalization_feature_design.json`, and `references/docs/generated/openapi.json`.

### Action
Read prior ATS label components, score-band policy, feature-quality blockers, and public API score constraints. Keep all Phase 7 decisions machine-readable.

### Expected output
Reusable variables for ATS component weights, score bands, OpenAPI score constraints, and final report writing.

### Verification
Fail fast if any required input is missing. Confirm public `atsFriendliness.score` remains constrained to integer `0-100` and ATS component weights sum to `1.0`.

In [19]:
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Any


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "TODOS.md").exists() and (candidate / "training").exists():
            return candidate
    raise RuntimeError("Could not locate repository root from notebook runtime.")


ROOT = find_repo_root(Path.cwd())
REPORTS = ROOT / "reports"
OPENAPI_PATH = ROOT / "references" / "docs" / "generated" / "openapi.json"
PHASE2_PATH = REPORTS / "phase_02_label_schema_baselines.json"
PHASE3_PATH = REPORTS / "phase_03_normalization_feature_design.json"
REPORT_PATH = REPORTS / "phase_07_ats_friendliness_scoring.json"


def read_json(path: Path) -> dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(f"Required Phase 7 input is missing: {path.relative_to(ROOT)}")
    return json.loads(path.read_text())


def get_schema(spec: dict[str, Any], ref: str) -> dict[str, Any]:
    if not ref.startswith("#/"):
        raise ValueError(f"Unsupported local ref: {ref}")
    node: Any = spec
    for part in ref[2:].split("/"):
        node = node[part]
    return node


def resolve_ref(spec: dict[str, Any], node: dict[str, Any]) -> dict[str, Any]:
    return get_schema(spec, node["$ref"]) if "$ref" in node else node


openapi = read_json(OPENAPI_PATH)
phase2 = read_json(PHASE2_PATH)
phase3 = read_json(PHASE3_PATH)

public_ats_schema = openapi["components"]["schemas"]["CvAnalysis"]["properties"]["analysisResult"]["properties"]["atsFriendliness"]
ats_score_schema = public_ats_schema["properties"]["score"]
assert ats_score_schema == {"type": "integer", "minimum": 0, "maximum": 100}

ats_components = phase2["ats_label_components"]
component_weight_sum = round(sum(float(component["weight"]) for component in ats_components), 10)
assert component_weight_sum == 1.0

score_band_policy = phase2["score_band_policy"]
setup_summary = {
    "public_ats_schema": public_ats_schema,
    "ats_score_schema": ats_score_schema,
    "ats_component_count": len(ats_components),
    "ats_component_weight_sum": component_weight_sum,
    "inherited_blockers": sorted(set(phase2.get("blocked_until_later_phases", []) + phase3.get("blocked_until_later_phases", []))),
}
setup_summary

{'public_ats_schema': {'type': 'object',
  'additionalProperties': False,
  'required': ['score', 'summary'],
  'properties': {'score': {'type': 'integer', 'minimum': 0, 'maximum': 100},
   'summary': {'type': 'string'}}},
 'ats_score_schema': {'type': 'integer', 'minimum': 0, 'maximum': 100},
 'ats_component_count': 6,
 'ats_component_weight_sum': 1.0,
 'inherited_blockers': ['Phase 10 must calibrate score bands before production score semantics are finalized.',
  'Phase 10 must calibrate score meanings after feature and label distributions are stable.',
  'Phase 3 must define normalization rules before labels are generated.',
  'Phase 4 must generate balanced pairs using these normalization policies and publish feature-quality metrics.',
  'Phase 4 must generate balanced pairs with leakage-safe splits before baseline evaluation.',
  'Phase 5 must compare baselines using these normalized features before complex model training.',
  'Phase 5 must run this baseline catalog before model t

## Step 7.1 — Benchmark coverage

### Purpose
Document required CV cases: normal PDF, scanned PDF, multi-column PDF, DOCX, table-heavy CV, very short CV, and overly long CV.

### Required input
Controlled CV samples with documented license/consent, parser output, file metadata, page count, extracted text length, section evidence, and reviewer labels. Raw contact PII must not be stored in benchmark labels.

### Action
Create a fixed benchmark-case catalog with stable case IDs, inclusion requirements, expected ATS risk, labels to collect, and acceptance gates. Use real CVs only when consent and anonymization are documented; otherwise use synthetic fixtures.

### Expected output
A machine-readable benchmark coverage plan for all required CV file/layout cases.

### Verification
Every required case from the TODO is present. Each case has required evidence, minimum bootstrap coverage, preferred coverage, and a blocker policy for missing samples.

In [20]:
benchmark_cases = [
    {
        "case_id": "normal_pdf",
        "file_family": "pdf",
        "layout_risk": "low",
        "required_evidence": ["text_layer_present", "page_count", "extracted_character_count", "section_heading_evidence"],
        "labels_to_collect": ["parseability_rating", "section_completeness_rating", "final_ats_band", "detected_issue_keys"],
        "minimum_bootstrap_cases": 10,
        "preferred_cases": 30,
        "expected_primary_issues": [],
        "acceptance_gate": "At least one high-quality, mostly parseable CV sample exists before score bucket agreement is claimed.",
    },
    {
        "case_id": "scanned_pdf",
        "file_family": "pdf",
        "layout_risk": "critical",
        "required_evidence": ["image_only_or_low_text_layer", "ocr_status", "extracted_character_count", "parser_warning"],
        "labels_to_collect": ["empty_parse_risk", "scanned_or_image_only", "final_ats_band", "reviewer_confidence"],
        "minimum_bootstrap_cases": 10,
        "preferred_cases": 30,
        "expected_primary_issues": ["scanned_or_image_only", "empty_text", "low_text_coverage"],
        "acceptance_gate": "Scanned/image-only files must be represented before empty-text precision or recall is claimed.",
    },
    {
        "case_id": "multi_column_pdf",
        "file_family": "pdf",
        "layout_risk": "high",
        "required_evidence": ["column_count_or_layout_signal", "text_order_sample", "section_boundary_evidence", "parser_warning"],
        "labels_to_collect": ["multi_column_risk", "parseability_rating", "section_completeness_rating", "final_ats_band"],
        "minimum_bootstrap_cases": 10,
        "preferred_cases": 30,
        "expected_primary_issues": ["multi_column_risk", "low_text_coverage"],
        "acceptance_gate": "Text-order failure examples must be reviewed before treating multi-column PDFs as low risk.",
    },
    {
        "case_id": "docx",
        "file_family": "docx",
        "layout_risk": "medium",
        "required_evidence": ["docx_parse_status", "paragraph_count", "table_count", "extracted_character_count"],
        "labels_to_collect": ["parseability_rating", "section_completeness_rating", "table_heavy_layout", "final_ats_band"],
        "minimum_bootstrap_cases": 10,
        "preferred_cases": 30,
        "expected_primary_issues": ["unsupported_file_type"],
        "acceptance_gate": "DOCX parser support must be explicit; unsupported DOCX cannot be silently scored high.",
    },
    {
        "case_id": "table_heavy_cv",
        "file_family": "pdf_or_docx",
        "layout_risk": "high",
        "required_evidence": ["table_density", "text_order_sample", "section_boundary_evidence", "parser_warning"],
        "labels_to_collect": ["table_heavy_layout", "parseability_rating", "section_completeness_rating", "final_ats_band"],
        "minimum_bootstrap_cases": 10,
        "preferred_cases": 30,
        "expected_primary_issues": ["table_heavy_layout", "low_text_coverage"],
        "acceptance_gate": "Table-heavy fixtures must include successful and failed parses to avoid over-penalizing all tables.",
    },
    {
        "case_id": "very_short_cv",
        "file_family": "pdf_or_docx_text",
        "layout_risk": "medium",
        "required_evidence": ["extracted_character_count", "word_count", "section_count", "experience_or_project_evidence"],
        "labels_to_collect": ["section_completeness_rating", "metric_evidence_rating", "final_ats_band", "detected_issue_keys"],
        "minimum_bootstrap_cases": 10,
        "preferred_cases": 30,
        "expected_primary_issues": ["missing_experience_section", "low_quantified_impact"],
        "acceptance_gate": "Short but valid entry-level CVs must be distinguished from empty/under-informative parses.",
    },
    {
        "case_id": "overly_long_cv",
        "file_family": "pdf_or_docx_text",
        "layout_risk": "medium",
        "required_evidence": ["page_count", "word_count", "section_count", "duplicate_section_or_repeated_text_signal"],
        "labels_to_collect": ["formatting_risk_rating", "section_completeness_rating", "final_ats_band", "reviewer_confidence"],
        "minimum_bootstrap_cases": 10,
        "preferred_cases": 30,
        "expected_primary_issues": ["excessive_length", "generic_responsibility_only"],
        "acceptance_gate": "Long-CV penalty must be based on reviewer evidence, not page count alone.",
    },
]

required_cases = {"normal_pdf", "scanned_pdf", "multi_column_pdf", "docx", "table_heavy_cv", "very_short_cv", "overly_long_cv"}
assert {case["case_id"] for case in benchmark_cases} == required_cases
benchmark_cases

[{'case_id': 'normal_pdf',
  'file_family': 'pdf',
  'layout_risk': 'low',
  'required_evidence': ['text_layer_present',
   'page_count',
   'extracted_character_count',
   'section_heading_evidence'],
  'labels_to_collect': ['parseability_rating',
   'section_completeness_rating',
   'final_ats_band',
   'detected_issue_keys'],
  'minimum_bootstrap_cases': 10,
  'preferred_cases': 30,
  'expected_primary_issues': [],
  'acceptance_gate': 'At least one high-quality, mostly parseable CV sample exists before score bucket agreement is claimed.'},
 {'case_id': 'scanned_pdf',
  'file_family': 'pdf',
  'layout_risk': 'critical',
  'required_evidence': ['image_only_or_low_text_layer',
   'ocr_status',
   'extracted_character_count',
   'parser_warning'],
  'labels_to_collect': ['empty_parse_risk',
   'scanned_or_image_only',
   'final_ats_band',
   'reviewer_confidence'],
  'minimum_bootstrap_cases': 10,
  'preferred_cases': 30,
  'expected_primary_issues': ['scanned_or_image_only',
   'empty

## Step 7.2 — Issue taxonomy

### Purpose
Define parseable text, section completeness, contact detection, date detection, metric evidence, excessive formatting risk, and empty parse risk.

### Required input
Phase 2 ATS label components, parser metadata, extracted CV text, anonymized boolean contact evidence, detected sections, date-parse evidence, metric/number evidence, and layout-risk signals.

### Action
Define each issue family with stable issue keys, source evidence, severity, score effect, privacy rule, and review rule. Keep issue keys separate from job-fit labels.

### Expected output
A durable ATS issue taxonomy that can drive rule scoring, reviewer labels, future classifier targets, and wrapper summaries.

### Verification
Every TODO issue family is represented. Contact evidence stores booleans only, not raw email, phone, or profile URLs.

In [21]:
issue_taxonomy = [
    {
        "family": "parseable_text",
        "component": "parseability",
        "issue_keys": ["low_text_coverage", "parser_warning", "unsupported_file_type"],
        "source_evidence": ["extracted_character_count", "word_count", "parser_status", "file_type"],
        "severity": "high",
        "score_effect": "Reduce parseability component; cap total score when text coverage is too low.",
        "privacy_rule": "Store aggregate extraction counts and parser status only.",
        "review_rule": "Manual review checks whether readable CV content was lost during parsing.",
    },
    {
        "family": "section_completeness",
        "component": "section_completeness",
        "issue_keys": ["missing_skills_section", "missing_experience_section", "missing_education_section", "ambiguous_section_headings"],
        "source_evidence": ["detected_section_keys", "heading_text_normalized", "section_count"],
        "severity": "medium",
        "score_effect": "Score by required section coverage; missing experience or skills is stronger than missing optional sections.",
        "privacy_rule": "Store section labels and counts, not full raw section text in labels.",
        "review_rule": "Reviewer confirms whether section content exists even when heading is nonstandard.",
    },
    {
        "family": "contact_detection",
        "component": "contact_detection",
        "issue_keys": ["missing_email", "missing_phone", "missing_portfolio_or_profile_link"],
        "source_evidence": ["has_email", "has_phone", "has_portfolio_or_profile_link"],
        "severity": "medium",
        "score_effect": "Missing all contact evidence is high severity; missing optional portfolio/profile link is role-dependent and lower severity.",
        "privacy_rule": "Store booleans only; never store raw contact values in training labels or reports.",
        "review_rule": "Reviewer checks detection false negatives on anonymized display, not persisted PII.",
    },
    {
        "family": "date_detection",
        "component": "date_detection",
        "issue_keys": ["missing_dates", "ambiguous_dates", "inconsistent_date_order"],
        "source_evidence": ["date_pattern_count", "date_parse_success_rate", "experience_entry_count"],
        "severity": "medium",
        "score_effect": "Score by parseable date coverage for experience and education entries.",
        "privacy_rule": "Store normalized date-pattern evidence and consistency flags only.",
        "review_rule": "Locale-specific date formats must be reviewed before marking dates missing.",
    },
    {
        "family": "metric_evidence",
        "component": "metric_evidence",
        "issue_keys": ["low_quantified_impact", "generic_responsibility_only"],
        "source_evidence": ["numeric_evidence_count", "percentage_or_currency_count", "impact_verb_context"],
        "severity": "low_to_medium",
        "score_effect": "Reward quantified achievements; do not invent metrics when absent.",
        "privacy_rule": "Store aggregate counts and anonymized snippets only when needed for audit.",
        "review_rule": "Reviewer distinguishes legitimate qualitative CVs from generic responsibility lists.",
    },
    {
        "family": "excessive_formatting_risk",
        "component": "formatting_risk",
        "issue_keys": ["table_heavy_layout", "multi_column_risk", "excessive_graphics", "text_order_risk"],
        "source_evidence": ["table_density", "column_count", "image_area_ratio", "text_order_sample"],
        "severity": "medium_to_high",
        "score_effect": "Subtract bounded formatting penalties; low-risk layout should not be penalized.",
        "privacy_rule": "Store layout metrics and parser diagnostics, not screenshots unless consented fixtures exist.",
        "review_rule": "Reviewer confirms whether layout actually harms ATS text order or section detection.",
    },
    {
        "family": "empty_parse_risk",
        "component": "parseability",
        "issue_keys": ["empty_text", "scanned_or_image_only", "ocr_required"],
        "source_evidence": ["extracted_character_count", "ocr_status", "image_only_signal", "parser_status"],
        "severity": "critical",
        "score_effect": "Hard cap total score at low band until OCR/parser recovery is proven.",
        "privacy_rule": "Store parser status and counts only.",
        "review_rule": "Every empty parse must enter failure-case review before release metrics are accepted.",
    },
]

required_issue_families = {"parseable_text", "section_completeness", "contact_detection", "date_detection", "metric_evidence", "excessive_formatting_risk", "empty_parse_risk"}
assert {issue["family"] for issue in issue_taxonomy} == required_issue_families
assert all("raw" not in key for issue in issue_taxonomy for key in issue.get("source_evidence", []))
issue_taxonomy

[{'family': 'parseable_text',
  'component': 'parseability',
  'issue_keys': ['low_text_coverage',
   'parser_warning',
   'unsupported_file_type'],
  'source_evidence': ['extracted_character_count',
   'word_count',
   'parser_status',
   'file_type'],
  'severity': 'high',
  'score_effect': 'Reduce parseability component; cap total score when text coverage is too low.',
  'privacy_rule': 'Store aggregate extraction counts and parser status only.',
  'review_rule': 'Manual review checks whether readable CV content was lost during parsing.'},
 {'family': 'section_completeness',
  'component': 'section_completeness',
  'issue_keys': ['missing_skills_section',
   'missing_experience_section',
   'missing_education_section',
   'ambiguous_section_headings'],
  'source_evidence': ['detected_section_keys',
   'heading_text_normalized',
   'section_count'],
  'severity': 'medium',
  'score_effect': 'Score by required section coverage; missing experience or skills is stronger than missing opt

## Step 7.3 — Scoring rule design

### Purpose
Describe a transparent rule-based baseline before any classifier is considered.

### Required input
Issue taxonomy, Phase 2 ATS component weights, parser metadata, extracted-text quality metrics, section/contact/date/metric/layout evidence, and manual labels for validation only.

### Action
Define a deterministic weighted score from component scores plus hard caps for critical parse failures. Keep the baseline explainable and auditable before any classifier or neural model is allowed.

### Expected output
A rule-based baseline spec with component weights, score formula, hard caps, detected issue mapping, confidence notes, and classifier eligibility gate.

### Verification
Weights sum to `1.0`, score range is `0-100`, critical empty-parse cases cannot receive medium/high scores, and no job-fit features are used.

In [22]:
component_weights = {component["component"]: float(component["weight"]) for component in ats_components}

rule_based_baseline = {
    "baseline_id": "ats_rule_baseline_v1",
    "score_formula": "round(100 * sum(component_score[component] * weight[component])) after applying hard caps",
    "score_range": {"minimum": 0, "maximum": 100, "type": "integer"},
    "component_weights": component_weights,
    "component_score_inputs": {
        "parseability": ["extracted_character_count", "word_count", "parser_status", "ocr_status", "text_layer_present"],
        "section_completeness": ["detected_required_sections", "section_count", "ambiguous_section_headings"],
        "contact_detection": ["has_email", "has_phone", "has_portfolio_or_profile_link"],
        "date_detection": ["date_parse_success_rate", "date_pattern_count", "inconsistent_date_order"],
        "metric_evidence": ["numeric_evidence_count", "percentage_or_currency_count", "impact_verb_context"],
        "formatting_risk": ["table_density", "column_count", "image_area_ratio", "text_order_risk", "file_type"],
    },
    "hard_caps": [
        {"condition": "empty_text == true", "max_score": 20, "issue_key": "empty_text"},
        {"condition": "scanned_or_image_only == true and ocr_status != 'successful'", "max_score": 30, "issue_key": "scanned_or_image_only"},
        {"condition": "unsupported_file_type == true", "max_score": 40, "issue_key": "unsupported_file_type"},
        {"condition": "low_text_coverage == true", "max_score": 55, "issue_key": "low_text_coverage"},
        {"condition": "no_contact_evidence == true", "max_score": 70, "issue_key": "missing_email_or_phone"},
    ],
    "detected_issue_rule": "Emit issue keys whose component evidence crosses documented thresholds; include severity and source_evidence keys, not raw PII.",
    "confidence_notes": [
        "unknown_layout_metrics lowers confidence but does not create a false issue",
        "manual_review_required for empty parses, parser errors, or conflicting reviewer labels",
        "score semantics remain draft until Phase 10 calibration",
    ],
    "excluded_features": [
        "jobFitAlignment.score",
        "job title",
        "job requirements",
        "recommendation rank",
        "wrapper-generated summaries",
        "topActionables",
        "sectionReviews",
    ],
    "classifier_eligibility_gate": {
        "minimum_cases_per_issue_family": 30,
        "requires_manual_labels": True,
        "requires_rule_baseline_report": True,
        "requires_privacy_review": True,
        "decision_now": "not_eligible_yet_rule_baseline_first",
    },
}

assert round(sum(rule_based_baseline["component_weights"].values()), 10) == 1.0
assert rule_based_baseline["score_range"] == {"minimum": 0, "maximum": 100, "type": "integer"}
assert any(cap["max_score"] <= 34 for cap in rule_based_baseline["hard_caps"] if cap["issue_key"] == "empty_text")
rule_based_baseline

{'baseline_id': 'ats_rule_baseline_v1',
 'score_formula': 'round(100 * sum(component_score[component] * weight[component])) after applying hard caps',
 'score_range': {'minimum': 0, 'maximum': 100, 'type': 'integer'},
 'component_weights': {'parseability': 0.3,
  'section_completeness': 0.2,
  'contact_detection': 0.15,
  'date_detection': 0.1,
  'metric_evidence': 0.15,
  'formatting_risk': 0.1},
 'component_score_inputs': {'parseability': ['extracted_character_count',
   'word_count',
   'parser_status',
   'ocr_status',
   'text_layer_present'],
  'section_completeness': ['detected_required_sections',
   'section_count',
   'ambiguous_section_headings'],
  'contact_detection': ['has_email',
   'has_phone',
   'has_portfolio_or_profile_link'],
  'date_detection': ['date_parse_success_rate',
   'date_pattern_count',
   'inconsistent_date_order'],
  'metric_evidence': ['numeric_evidence_count',
   'percentage_or_currency_count',
   'impact_verb_context'],
  'formatting_risk': ['table_d

## Step 7.4 — Evaluation policy

### Purpose
Define precision, recall, score-bucket agreement, empty-text rate, and failure-case review workflow.

### Required input
Locked benchmark manifest, manual reviewer labels, rule-baseline predictions, detected issue keys, score bands, parser statuses, and file-case metadata.

### Action
Define evaluation metrics and review workflow before implementation. Metrics must be reported globally and by benchmark case family. Empty-text failures must be reviewed, not hidden in aggregate scores.

### Expected output
An evaluation policy with metric definitions, required slices, minimum coverage, failure-case workflow, and promotion blockers.

### Verification
Precision and recall are defined per issue key. Score-bucket agreement uses the same low/medium/high bands as Phase 2. Empty-text rate is reported separately from overall accuracy.

In [23]:
evaluation_policy = {
    "label_source": "manual reviewer labels on locked benchmark manifest; weak parser labels may bootstrap only after review",
    "score_bands": [
        {
            "band": band["band"],
            "api_min": band["api_min"],
            "api_max": band["api_max"],
            "product_meaning_ats": band["product_meaning_ats"],
        }
        for band in score_band_policy
    ],
    "metrics": [
        {"metric": "issue_precision", "formula": "true_positive_issue / predicted_issue", "level": "per_issue_key_and_macro_average"},
        {"metric": "issue_recall", "formula": "true_positive_issue / labeled_issue", "level": "per_issue_key_and_macro_average"},
        {"metric": "score_bucket_agreement", "formula": "predicted_low_medium_high == reviewer_low_medium_high", "level": "global_and_by_case_family"},
        {"metric": "empty_text_rate", "formula": "empty_text_predictions / total_files", "level": "global_by_parser_and_case_family"},
        {"metric": "critical_failure_recall", "formula": "detected_empty_or_scanned_or_unsupported / labeled_critical_parse_failures", "level": "critical_issue_family"},
        {"metric": "review_disagreement_rate", "formula": "reviewer_disagreements / double_reviewed_cases", "level": "global_and_by_issue_family"},
    ],
    "required_slices": ["case_id", "file_family", "parser_status", "language", "score_band", "issue_family"],
    "minimum_coverage": {
        "bootstrap_cases_per_required_case": 10,
        "preferred_cases_per_required_case": 30,
        "double_review_minimum_share": 0.2,
        "critical_failure_review_share": 1.0,
    },
    "failure_case_review_workflow": [
        "Queue every empty-text, scanned/image-only, unsupported-file, and parser-error case for manual review.",
        "For each false negative issue, record missing source signal, parser limitation, and taxonomy update need.",
        "For each false positive issue, record threshold problem, layout exception, or reviewer ambiguity.",
        "Publish aggregate review findings without raw contact values or full CV text.",
        "Block promotion if critical parse failures are not reviewed or if score-bucket errors cluster in one required case family.",
    ],
    "promotion_policy": {
        "current_decision": "GO for rule design and benchmark policy; NO-GO for production ATS scoring claim until benchmark labels and calibration exist",
        "must_report_before_classifier": ["rule_baseline_metrics", "per_issue_precision_recall", "score_bucket_agreement", "empty_text_rate", "failure_case_review"],
        "phase10_dependency": "Production score semantics and thresholds require calibration diagnostics before release claims.",
    },
}

metric_names = {metric["metric"] for metric in evaluation_policy["metrics"]}
assert {"issue_precision", "issue_recall", "score_bucket_agreement", "empty_text_rate"}.issubset(metric_names)
evaluation_policy

{'label_source': 'manual reviewer labels on locked benchmark manifest; weak parser labels may bootstrap only after review',
 'score_bands': [{'band': 'low',
   'api_min': 0,
   'api_max': 34,
   'product_meaning_ats': 'CV has serious parseability, structure, contact/date, metric, or formatting issues that can block reliable screening.'},
  {'band': 'medium',
   'api_min': 35,
   'api_max': 64,
   'product_meaning_ats': 'CV is mostly parseable but has issues that can reduce ranking or hide important evidence.'},
  {'band': 'high',
   'api_min': 65,
   'api_max': 100,
   'product_meaning_ats': 'CV is parseable, structured, contact/date evidence is detectable, and formatting risk is low.'}],
 'metrics': [{'metric': 'issue_precision',
   'formula': 'true_positive_issue / predicted_issue',
   'level': 'per_issue_key_and_macro_average'},
  {'metric': 'issue_recall',
   'formula': 'true_positive_issue / labeled_issue',
   'level': 'per_issue_key_and_macro_average'},
  {'metric': 'score_bucket

## Step 7.5 — Output contract

### Purpose
Define model/core output fields: `atsFriendliness.score` and `atsFriendliness.detectedIssues`.

### Required input
OpenAPI public `atsFriendliness` score/summary shape, rule-baseline score, detected issue keys, source evidence keys, and wrapper boundary from the training scope.

### Action
Define core ATS output fields, grounding rules, type/range constraints, public API mapping, and fields that remain outside model ownership.

### Expected output
A durable output contract and machine-readable Phase 7 report.

### Verification
The core output contains score and detected issues, stays separate from job-fit scoring, maps safely to the public API, and excludes wrapper/backend-owned fields.

In [24]:
output_contract = {
    "schema_version": "ats-friendliness-core-v1",
    "model_owned_fields": [
        {
            "field": "score",
            "public_path": "atsFriendliness.score",
            "type": "integer",
            "range": [0, 100],
            "grounding_rule": "Derived only from ATS component evidence and Phase 10 calibration; never from job-fit score, recommendation rank, or wrapper copy.",
        },
        {
            "field": "detectedIssues",
            "public_path": "core_only.atsFriendliness.detectedIssues",
            "type": "array[object]",
            "range": "0-20 issues",
            "grounding_rule": "Issue keys must come from the Phase 7 taxonomy and include severity plus source_evidence_keys without raw PII.",
            "item_shape": {
                "issueKey": "stable taxonomy key",
                "family": "parseable_text | section_completeness | contact_detection | date_detection | metric_evidence | excessive_formatting_risk | empty_parse_risk",
                "severity": "low | medium | high | critical",
                "sourceEvidenceKeys": "array[string]",
            },
        },
        {
            "field": "componentScores",
            "public_path": "core_only.atsFriendliness.componentScores",
            "type": "object",
            "range": "component score values from 0.0 to 1.0",
            "grounding_rule": "Debug/audit field for model card and calibration; not required in public API response.",
        },
        {
            "field": "confidenceNotes",
            "public_path": "core_only.atsFriendliness.confidenceNotes",
            "type": "array[string]",
            "range": "0-5 diagnostic notes",
            "grounding_rule": "Notes describe data quality, parser uncertainty, missing layout metrics, or manual-review requirement.",
        },
    ],
    "public_api_mapping": {
        "atsFriendliness.score": "core.score rounded/clipped to integer 0-100 after Phase 10 calibration",
        "atsFriendliness.summary": "API wrapper renders concise product copy from detectedIssues, score band, and confidenceNotes",
    },
    "not_model_owned_fields": [
        "topActionables",
        "sectionReviews",
        "overallImpression",
        "jobFitAlignment",
        "jobRecommendations",
        "auth",
        "persistence",
        "file storage",
        "raw contact values",
    ],
    "separation_policy": {
        "ats_vs_jobfit": "ATS score uses CV quality and parser/layout evidence only; job-fit score uses candidate/job alignment evidence only.",
        "summary_boundary": "Core emits issue signals; wrapper writes public summary without adding unsupported claims.",
        "privacy_boundary": "Detected issues can mention missing contact evidence but must not expose raw email, phone, or URL values.",
    },
}

assert any(field["field"] == "score" for field in output_contract["model_owned_fields"])
assert any(field["field"] == "detectedIssues" for field in output_contract["model_owned_fields"])
assert output_contract["public_api_mapping"]["atsFriendliness.score"].startswith("core.score")

phase7_report = {
    "schema_version": "phase-07-ats-friendliness-scoring-v1",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "inputs": {
        "openapi_json": str(OPENAPI_PATH.relative_to(ROOT)),
        "phase2_report": str(PHASE2_PATH.relative_to(ROOT)),
        "phase3_report": str(PHASE3_PATH.relative_to(ROOT)),
    },
    "setup_summary": setup_summary,
    "benchmark_cases": benchmark_cases,
    "issue_taxonomy": issue_taxonomy,
    "rule_based_baseline": rule_based_baseline,
    "evaluation_policy": evaluation_policy,
    "output_contract": output_contract,
    "blocked_until_later_phases": [
        "Production ATS score semantics require Phase 10 calibration evidence.",
        "Classifier training is blocked until benchmark labels exist with enough examples per issue family.",
        "Release claims are blocked until empty-text and critical parse failures receive manual review.",
    ],
    "acceptance": {
        "ats_scoring_is_separate_from_job_fit_scoring": True,
        "supported_file_cases_are_documented": True,
        "evaluation_metrics_are_defined_before_implementation": True,
    },
}

REPORTS.mkdir(parents=True, exist_ok=True)
REPORT_PATH.write_text(json.dumps(phase7_report, indent=2, sort_keys=True) + "\n")
{
    "report_path": str(REPORT_PATH.relative_to(ROOT)),
    "benchmark_case_count": len(benchmark_cases),
    "issue_family_count": len(issue_taxonomy),
    "acceptance": phase7_report["acceptance"],
}

{'report_path': 'reports/phase_07_ats_friendliness_scoring.json',
 'benchmark_case_count': 7,
 'issue_family_count': 7,
 'acceptance': {'ats_scoring_is_separate_from_job_fit_scoring': True,
  'supported_file_cases_are_documented': True,
  'evaluation_metrics_are_defined_before_implementation': True}}

## Acceptance criteria

- [x] ATS scoring is separate from job-fit scoring.
- [x] Supported file cases are documented.
- [x] Evaluation metrics are defined before implementation.

## Phase notes

- Phase 7 completes the ATS friendliness benchmark, taxonomy, rule baseline, evaluation policy, and core output contract.
- No production ATS classifier is trained in this phase. A transparent rule baseline and controlled benchmark labels must exist first.
- Production score semantics remain blocked until Phase 10 calibration evidence exists.
- Public API exposes `atsFriendliness.score` and `atsFriendliness.summary`; model/core may emit `detectedIssues` for wrapper rendering and audit.